### **LangChain: LangChain: Models, Prompts and Output Parsers**
#### **Outline**
- Direct API calls to OpenAI
- API calls through LangChain:
  - Prompts
  - Models
  - Output parsers

#### **Get [OpenAI_API_KEY](https://platform.openai.com/api-keys)**

In [21]:
#!pip install python-dotenv
#!pip install openai

In [1]:
import os, openai
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
openai.api_key = os.environ['OPENAI_API_KEY']

llm_model = 'gpt-3.5-turbo'

#### **Chat API : OpenAI**

In [10]:
def get_completion(prompt, model = llm_model):
    messages = [{'role': 'user', 'content': prompt}]
    response = openai.chat.completions.create(
        model = model,
        messages = messages,
        temperature = 0
    )
    # Can replace 0 by a natural number, as you want more answers
    return response.choices[0].message.content 

# get_completion('What is 1+1?')

In [11]:
customer_email = '''
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
'''

customer_style = '''
American English \
in a calm and respectful tone
'''

prompt = f'''
Translate the text \
that is delimited by triple backticks 
into a style that is {customer_style}.
Text: ```{customer_email}```
'''

#### **Chat API : LangChain**

In [31]:
#!pip install --upgrade langchain

#### **Model**

In [12]:
from langchain.chat_models import ChatOpenAI

# For no randomness and creativity of the generated, text by an LLM, set temperature = 0.0
chat = ChatOpenAI(temperature=0.0, model=llm_model)

#### **Prompt template**

In [13]:
template_string = '''
Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{email}```
'''

In [ ]:
from langchain.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)
customer_messages = prompt_template.format_messages(
                    style = customer_style,
                    email = customer_email
                    )

# Currently, customer_messages only has 1 object. Thus,..
# type(customer_messages) is list
# len(customer_messages) is 1
#
# To extend the list, we can do..
from langchain.schema import HumanMessage, SystemMessage
extended_messages = [
    customer_messages[0], # `HumanMessage` by default
    SystemMessage(content = f"I am from the System. {customer_style}")
]

# Call the LLM to translate to the style of the customer message
# customer_response = chat(customer_messages)

#### **Output Parsers**
Define the LLM output

In [ ]:
{
  'gift': False,
  'delivery_days': 5,
  'price_value': 'pretty affordable!'
}

In [16]:
customer_review = '''\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
'''

review_template = '''\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
'''

In [17]:
prompt_template = ChatPromptTemplate.from_template(review_template)

customer_messages = prompt_template.format_messages(text = customer_review)

# customer_response = chat(messages)

#### **Parse the LLM output string into a Python dictionary**

In [22]:
from langchain.output_parsers import ResponseSchema
from langchain.output_parsers import StructuredOutputParser

gift_schema = ResponseSchema(name = 'gift',
                            description = 'Was the item purchased as a gift for someone else? \
                            Answer True if yes, False if not or unknown.')

delivery_days_schema = ResponseSchema(name ='delivery_days',
                                    description = 'How many days did it take for the product \
                                    to arrive? If this information is not found, output -1.')

price_value_schema = ResponseSchema(name = 'price_value',
                                    description = 'Extract any sentences about the value or \
                                    price, and output them as a comma separated Python list.')

response_schemas = [gift_schema, 
                    delivery_days_schema,
                    price_value_schema]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [ ]:
review_template_2 = '''\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
'''

prompt_template = ChatPromptTemplate.from_template(review_template_2)

customer_messages = prompt_template.format_messages(text = customer_review, 
                                                    format_instructions = format_instructions)

# customer_response = chat(customer_messages)
# output_dict = output_parser.parse(customer_response.content)
# output_dict.get('delivery_days')